# Gate Vision: teach the plate reader Pakistani plates

This runs free on Google Colab. Press **Runtime > Run all** and wait (about 20 to 40 minutes).

**Before you start**
1. **Runtime > Change runtime type > T4 GPU** (free). It also works without a GPU, just slower.
2. In your Google Drive make a folder called **GateVisionTraining** and put in it:
   - **plates.zip**: made on the server by `training\pack_plates.bat` (your labelled gate photos). *This matters most.*
   - optional: a folder **public** with Roboflow dataset zips (export format "YOLOv8"), for example
     *Pakistan License Plate Detection* (MIT licence). See `training/README.md`.

**At the end** two files appear in GateVisionTraining: `pk_plate_ocr.onnx` and `pk_plate_ocr_config.yaml`,
plus a before/after score. Copy both files into `G:\gate-vision\models\` only if the score went up.

In [ ]:
# 1. Connect Google Drive and get the Gate Vision code
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/GateVisionTraining'
BRANCH = 'main'
!rm -rf /content/gv && git clone -q --depth 1 -b $BRANCH https://github.com/jahantch-maker/face-and-number-plate-detection-with-cctv /content/gv
!pip -q install "fast-plate-ocr[train,onnx]" fast-alpr onnx pyyaml
import os; os.chdir('/content/gv'); os.environ['KERAS_BACKEND'] = 'tensorflow'
print('ready')

In [ ]:
# 2. Prepare the photos: our labelled gate photos + public datasets + made-up bike plates
import glob, os, zipfile, subprocess
W = '/content/work'
!rm -rf $W && mkdir -p $W/own $W/public
own_zip = f'{DRIVE}/plates.zip'
own_labels = None
if os.path.exists(own_zip):
    zipfile.ZipFile(own_zip).extractall(f'{W}/own')
    found = glob.glob(f'{W}/own/**/labels.csv', recursive=True)
    own_labels = found[0] if found else None
print('own gate photos:', own_labels or 'NOT FOUND - training only on public and made-up plates')

public = []
for i, z in enumerate(sorted(glob.glob(f'{DRIVE}/public/*.zip'))):
    src, dst = f'{W}/public/src{i}', f'{W}/public/conv{i}'
    zipfile.ZipFile(z).extractall(src)
    r = subprocess.run(['python', 'training/convert_char_boxes.py', src, dst], capture_output=True, text=True)
    print(os.path.basename(z), '->', (r.stdout or r.stderr).strip().splitlines()[-1])
    if os.path.exists(f'{dst}/labels.csv'):
        public.append(f'{dst}/labels.csv')

!python training/synth_plates.py $W/synth --count 6000
args = ['python', 'training/make_dataset.py', '--out', f'{W}/ds', '--public', f'{W}/synth/labels.csv']
for p in public: args += ['--public', p]
if own_labels: args += ['--own', own_labels]
print(subprocess.run(args, capture_output=True, text=True).stdout)

In [ ]:
# 3. Download the reader Gate Vision uses today (we improve it rather than start from zero)
BASE = 'https://github.com/ankandrew/cnn-ocr-lp/releases/download/arg-plates'
!mkdir -p $W/base && cd $W/base && wget -q $BASE/cct_xs_v1_global.keras $BASE/cct_xs_v1_global_plate_config.yaml https://raw.githubusercontent.com/ankandrew/fast-plate-ocr/master/models/cct_xs_v1.yaml && ls

In [ ]:
# 4. Score the CURRENT reader on the check photos (never used for training)
!python training/evaluate_ocr.py --labels $W/ds/val.csv --model cct-xs-v1-global-model --out $W/before.csv
import os
if os.path.getsize(f'{W}/ds/val_bikes.csv') > 30:
    !python training/evaluate_ocr.py --labels $W/ds/val_bikes.csv --model cct-xs-v1-global-model --out $W/before_bikes.csv

In [ ]:
# 5. Train (stops by itself when it stops improving)
!rm -rf $W/out && fast-plate-ocr train \
  --model-config-file $W/base/cct_xs_v1.yaml \
  --plate-config-file $W/base/cct_xs_v1_global_plate_config.yaml \
  --weights-path $W/base/cct_xs_v1_global.keras \
  --annotations $W/ds/train.csv --val-annotations $W/ds/val.csv \
  --epochs 120 --batch-size 64 --lr 0.0005 --early-stopping-patience 20 \
  --output-dir $W/out

In [ ]:
# 6. Turn the result into the file Gate Vision loads, and score it on the same check photos
import glob, os, shutil
run = sorted(glob.glob(f'{W}/out/*/'))[-1]
!fast-plate-ocr export --model {run}best.keras --plate-config-file $W/base/cct_xs_v1_global_plate_config.yaml --format onnx --skip-validation > /dev/null 2>&1
!python training/fix_onnx.py {run}best.onnx --keras {run}best.keras --plate-config $W/base/cct_xs_v1_global_plate_config.yaml
os.makedirs(f'{W}/final', exist_ok=True)
shutil.copy(f'{run}best.onnx', f'{W}/final/pk_plate_ocr.onnx')
shutil.copy(f'{W}/base/cct_xs_v1_global_plate_config.yaml', f'{W}/final/pk_plate_ocr_config.yaml')
!python training/evaluate_ocr.py --labels $W/ds/val.csv --model $W/final/pk_plate_ocr.onnx --out $W/after.csv
if os.path.getsize(f'{W}/ds/val_bikes.csv') > 30:
    !python training/evaluate_ocr.py --labels $W/ds/val_bikes.csv --model $W/final/pk_plate_ocr.onnx --out $W/after_bikes.csv

In [ ]:
# 7. Save to Google Drive
import shutil, glob
for f in glob.glob(f'{W}/final/*') + glob.glob(f'{W}/before*.csv') + glob.glob(f'{W}/after*.csv'):
    shutil.copy(f, DRIVE)
print('Saved to Google Drive / GateVisionTraining:')
print('  pk_plate_ocr.onnx and pk_plate_ocr_config.yaml  -> copy both into G:\\gate-vision\\models\\ if the AFTER score is higher')
print('  before.csv / after.csv                            -> every check photo with what each reader read')